In [ ]:
import pandas as pd
import pyxdf
import numpy as np
import mne
import os
from IPython.display import display
from pathlib import Path

In [ ]:
"""
In this notebook, we preprocess and ICA the raw EEG. We then epoch them around sonication time (-2 to 7 seconds) and compute the time dependent PSD. 
"""

In [ ]:
xdf_path = Path(r"C:\Users\jshin\OW_closedloopLIFU\xdf_data\sub-dave_run_2\ses-1\eeg\sub-dave_run_2_ses-1_task-dave_run_2_run-001_eeg.xdf")
data, header = pyxdf.load_xdf(str(xdf_path))
for i, stream in enumerate(data):
    print(f"Stream {i}: {stream['info']['name'][0]}")

In [ ]:
EEG_LIFU_events = data[0]

# Create DataFrame from time_series
markers = pd.DataFrame(EEG_LIFU_events['time_series'])

# Rename the first column to 'markers'
markers.rename(columns={0: 'markers'}, inplace=True)

# Add timestamp column
markers['Timestamp'] = EEG_LIFU_events['time_stamps']

# Move Timestamp to be the first column after the index
cols = ['Timestamp'] + [col for col in markers.columns if col != 'Timestamp']
markers = markers[cols]
markers

In [ ]:
trial_ts = markers[markers['markers']=='LIFU_ON']['Timestamp']
trial_ts

In [ ]:
stream = data[2]
df= []
df = pd.DataFrame(stream['time_series'])
df = df.rename(columns={i: f"Ch{i}" for i in range(df.shape[1])})

# Add timestamp column
df['Timestamp'] = stream['time_stamps']

# Move Timestamp to be the first column after the index
cols = ['Timestamp'] + [col for col in df.columns if col != 'Timestamp']
df = df[cols]
eeg_raw = df[['Ch0', 'Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6', 'Ch12','Timestamp']]
eeg_raw

In [ ]:
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt

# first_data: rows = samples, columns = 8 EEG channels
# 1. Clean and Load Data
full_eeg_df = eeg_raw.drop(columns=["Ch12", "Timestamp"])
df = pd.DataFrame(full_eeg_df.copy())
df = df.replace([np.inf, -np.inf], np.nan)
df = df.replace(-200000.0, np.nan)
threshold = len(df) * 0.5
df = df.dropna(thresh=threshold, axis=1)
df = df.interpolate(method='linear', limit=5, limit_direction='both')
df = df.fillna(method='bfill').fillna(method='ffill')

# 2. MNE Array Building
sfreq = 250
data = df.values.T  # shape: (n_channels, n_samples)

# Mapping
ch_names = [
    "FCz",   # 1
    "CP3",   # 2
    "P5",  # 3
    "Cz",   # 4
    "Pz",   # 5
    "POz",  # 6
    "CP4",  # 7
    #"F5"    # 8 --> trigger channel
]

info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
raw = mne.io.RawArray(data, info)

# 3. ICA topography
montage = mne.channels.make_standard_montage("standard_1020")
raw.set_montage(montage)

# 4. Filtering
raw.filter(1., 40., fir_design='firwin')
raw.notch_filter(60.)
raw.set_eeg_reference('average')

# 5. ICA
ica = mne.preprocessing.ICA(
    n_components= 0.99999,
    random_state=97,
    max_iter='auto'
)
ica.fit(raw)

#plotting components
ica.plot_components()      # scalp maps
ica.plot_sources(raw)      # time series

ica.exclude = [0,5]  # manually add
raw_clean = ica.apply(raw.copy())



In [ ]:
trial_ts

In [ ]:
import numpy as np
import pandas as pd

# finding index of nearest timestamp for mne object

timestamps = eeg_raw['Timestamp'].values
trial_ts = np.asarray(trial_ts)

insert_idx = np.searchsorted(timestamps, trial_ts)
insert_idx = np.clip(insert_idx, 1, len(timestamps) - 1)

left_vals = timestamps[insert_idx - 1]
right_vals = timestamps[insert_idx]

use_left = np.abs(trial_ts - left_vals) < np.abs(trial_ts - right_vals)
matched_idx = np.where(use_left, insert_idx - 1, insert_idx)

matched_time = timestamps[matched_idx]
time_diff = matched_time - trial_ts  # negative = eeg sample before event, positive = after

fs = 1 / np.median(np.diff(timestamps))  # estimate actual sampling rate from data
expected_max_diff = 1 / fs  # one sample

n_bad = np.sum(np.abs(time_diff) > expected_max_diff)
if n_bad > 0:
    print(f"Warning: {n_bad} of {len(trial_ts)} matches exceed half a sample period "
          f"({expected_max_diff*1000:.2f} ms). Check for stream dropouts or clock offset.")
else:
    print(f"All {len(trial_ts)} matches within expected tolerance "
          f"({expected_max_diff*1000:.2f} ms), estimated fs = {fs:.2f} Hz.")

trial_index_df = pd.DataFrame({
    'trial_ts': trial_ts,
    'matched_index': matched_idx,
    'matched_timestamp': matched_time,
    'time_diff_sec': time_diff,
})
trial_index_df = trial_index_df.drop([1, 3]) # did not sonicate
trial_index_df

In [ ]:
event_samples = np.array(trial_index_df['matched_index'])

# Build MNE event array: shape (n_events, 3)
# Columns: [sample_index, 0, event_id]
events = np.column_stack([event_samples, 
                          np.zeros_like(event_samples, dtype=int), 
                          np.ones_like(event_samples, dtype=int)])
tmin = -2.0
tmax = 7.0

epochs = mne.Epochs(
    raw,
    events,
    event_id=1,
    tmin=tmin,
    tmax=tmax,
    baseline=(-0.2, 0),     
    preload=True
)


In [ ]:
from mne.time_frequency import stft
import numpy as np
import matplotlib.pyplot as plt

sfreq = epochs.info['sfreq']
ch_names = epochs.ch_names

all_spectrograms = []
all_times = []
all_freqs = []

# Loop over epochs
for idx, ep in enumerate(epochs):
    data = ep #* 1e-6   # µV → V
    n_fft = 256
    step = 64

    # stft
    Zxx = stft(data, wsize=n_fft, tstep=step)
    power = np.abs(Zxx)**2 / (n_fft**2) # normalizing

    # time axis
    tmin = epochs.tmin
    tmax = epochs.tmax
    times = np.linspace(tmin, tmax, power.shape[-1])
    # freq axis
    freqs = np.linspace(0, sfreq/2, power.shape[1])

    # frequencies you want to see (1-40Hz)
    mask = (freqs >= 1) & (freqs <= 40)
    freqs_1_40 = freqs[mask]
    power_1_40 = power[:, mask, :]

    # single graph for each epoch (averaging across channels)
    psd_avg = power_1_40.mean(axis=0)

    # convert to db for better visualizations
    psd_db = 10 * np.log10(psd_avg + 1e-20)

    # only use 95%
    vmin = np.percentile(psd_db, 5)
    vmax = np.percentile(psd_db, 95)
    
    all_spectrograms.append(psd_db)
    all_times.append(times)
    all_freqs.append(freqs_1_40)

    # plotting
    plt.figure(figsize=(10,6))
    plt.imshow(
        psd_db,
        aspect='auto',
        origin='lower',
        extent=[times[0], times[-1], freqs_1_40[0], freqs_1_40[-1]],
        vmin=vmin,
        vmax=vmax,
        cmap='viridis'
    )
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")
    plt.title(f"Epoch {idx} — Sonicating Time‑Resolved PSD (1–40 Hz, dB)")
    plt.colorbar(label="Power (dB)")
    plt.show()


In [ ]:
# PSD DATA ANALYSIS
# windowing 

fs = 250
prev = 5
pre_window = int(prev * fs)
# epochs run tmin=-5 to tmax=10 (15s total), so "post" is the remaining ~10s after t=0

n_epochs = epochs.get_data().shape[0]
pre_segments = []
post_segments = []
for i in range(n_epochs):
    eeg_filt = pd.DataFrame(epochs.get_data()[i].copy()).T  # time x channels for trial i
    pre_segments.append(eeg_filt.iloc[:pre_window])
    post_segments.append(eeg_filt.iloc[pre_window:])

In [ ]:
from scipy.signal import welch
import numpy as np

def compute_psd(seg):
    f, psd = welch(seg, fs=fs, nperseg=fs, axis=0)
    return f, psd  # psd shape: freqs × channels
pre_psds  = []
post_psds = []

for pre, post in zip(pre_segments, post_segments):
    f, pre_psd  = compute_psd(pre)
    f, post_psd = compute_psd(post)
    pre_psds.append(pre_psd)
    post_psds.append(post_psd)
pre_psd_mean  = np.mean(pre_psds, axis=0)
post_psd_mean = np.mean(post_psds, axis=0)


In [ ]:
def band_power(psd, freqs, f_lo, f_hi):
    idx = (freqs >= f_lo) & (freqs <= f_hi)
    return np.trapezoid(psd[idx, :], freqs[idx], axis=0)

bands = {"Theta": (4, 7), "Alpha": (8, 12), "Beta": (13, 30), "Gamma": (30, 40)}
ch_index = [f"Ch{i+1}" for i in range(7)]

# per-trial band power/percent-change/dB-change (pre_psds/post_psds hold one PSD per trial)
trial_pct = {name: [] for name in bands}
trial_db = {name: [] for name in bands}
for pre_psd, post_psd in zip(pre_psds, post_psds):
    for name, (lo, hi) in bands.items():
        pre_bp = band_power(pre_psd, f, lo, hi)
        post_bp = band_power(post_psd, f, lo, hi)
        trial_pct[name].append((post_bp - pre_bp) / pre_bp * 100)
        trial_db[name].append(10 * np.log10(post_bp / pre_bp))

# mean across trials -- this replaces the old single-trial percent change
bands_pct_df = pd.DataFrame({f"{name} %Δ": np.mean(trial_pct[name], axis=0) for name in bands}, index=ch_index)

# std across trials -- large values here mean the mean is being driven by one outlier
# trial rather than a consistent effect, which is exactly what happened before this fix
bands_pct_std_df = pd.DataFrame({f"{name} %Δ (trial std)": np.std(trial_pct[name], axis=0) for name in bands}, index=ch_index)

# dB change -- same log scaling as the STFT/Morlet plots, much less sensitive to a
# noisy/small pre-window denominator than the raw percent change above
bands_db_df = pd.DataFrame({f"{name} ΔdB": np.mean(trial_db[name], axis=0) for name in bands}, index=ch_index)

print("Percent change in band power (mean across all trials):")
display(bands_pct_df)

print("\nStd of percent change across trials (large = unreliable / outlier-driven):")
display(bands_pct_std_df)

print("\ndB change in band power (log-ratio, comparable to STFT/Morlet):")
display(bands_db_df)